# 🥈 Silver Macro-Sentiment Analysis & Market Timing Model

## 🎯 Project Goal
To analyze how **Global Macro-economic news** and **Industrial demand sentiment** influence the price of Silver (XAG), and to build a systematic trading strategy based on these signals.

---

## 💡 Research Question & Hypothesis
* **Primary Question:** Can news sentiment regarding inflation, industrial manufacturing (PMI), and Fed policy predict Silver price movements?
* **Hypothesis:** Silver acts as a hybrid asset. Bullish sentiment in "Green Energy" (Solar/EV) and Bearish sentiment in "Real Interest Rates" create a powerful synergy that leads to **Abnormal Returns** in Silver prices.

---

## 🛠️ Tech Stack & Data Sources

| Category | Tools / Sources |
| :--- | :--- |
| **News Data** | *Reuters, CNBC (Commodities section), Bloomberg, Mining.com*. |
| **Market Data** | `yfinance` (Ticker: `SI=F` for Futures or `SLV` for ETF), FRED API (Real Yields, CPI). |
| **NLP** | `Hugging Face Transformers`, **FinBERT** (Specialized for financial sentiment). |
| **Quant Analysis** | `Pandas`, `NumPy`, `Statsmodels` (Gold-Silver Ratio, Correlation Analysis). |
| **Backtesting** | `VectorBT` (High-performance event-driven backtesting). |

---

## 📋 Methodology (Step-by-Step)

### 📂 Phase 1: Data Pipeline & Engineering
1.  **Commodity News Ingestion:** Scrape or fetch news headlines using keywords: *Silver price, Silver demand, Federal Reserve, Solar Energy, Industrial PMI*.
2.  **Market Data Sync:** Fetch Silver Spot/Futures prices and align with Macro indicators (USD Index, Gold Price).

### 🧠 Phase 2: Hybrid Sentiment Scoring
3.  **Sentiment Scoring:** Use **FinBERT** to classify news into Positive/Negative.
4.  **Feature Engineering:** * **Monetary Sentiment:** Sentiment related to Inflation/Fed.
    * **Industrial Sentiment:** Sentiment related to Manufacturing/Green Tech.
    * **Gold-Silver Ratio (GSR):** Use as a mean-reversion feature.

### 🔬 Phase 3: Quantitative Validation
5.  **Event Study:** Analyze Silver's reaction to **FOMC meetings** and **Manufacturing PMI releases**.
6.  **Inter-market Analysis:** Test the Lead-Lag relationship between Gold sentiment and Silver price.
7.  **Information Coefficient (IC):** Measure the predictive power of "Industrial Sentiment" on Silver's 5-day forward return.

### 📈 Phase 4: Strategy & Backtesting
8.  **Silver Momentum Strategy:** Long Silver when both Monetary and Industrial sentiments are aligned positive.
9.  **Backtesting:** Evaluate using `VectorBT` considering Silver's high volatility and spread.

---

## 📊 Performance Metrics
* **Sharpe & Sortino Ratio:** Crucial for Silver due to its high volatility.
* **Beta to Gold:** Measure how much of the return is independent of Gold's movement.
* **Maximum Drawdown (MDD):** Assess risk during commodity market sell-offs.

---

## ⚠️ Risks & Challenges
* **Extreme Volatility:** Silver is known as "The Devil's Metal" for its violent price swings.
* **Industrial Substitution:** Risks of industries finding cheaper alternatives to silver.

## 1. Data and News Preparation

In [12]:
import yfinance as yf
import pandas as pd
import requests
from IPython.display import display

In [13]:
start_date = "2015-01-01"
end_date = "2025-01-01"

def get_silver_data(start=start_date, end=end_date):
    # SI=F is Silver Futures on COMEX
    silver = yf.download("SI=F", start=start, end=end)
    silver.index = pd.to_datetime(silver.index)
    print(f"Downloaded {len(silver)} days of Silver Futures data")
    return silver

In [14]:
df_prices = get_silver_data()
print(df_prices)

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  1 of 1 completed

Downloaded 2513 days of Silver Futures data
Price           Close       High        Low       Open Volume
Ticker           SI=F       SI=F       SI=F       SI=F   SI=F
Date                                                         
2015-01-02  15.734000  15.815000  15.535000  15.790000     13
2015-01-05  16.179001  16.179001  16.179001  16.179001      0
2015-01-06  16.603001  16.603001  16.603001  16.603001      2
2015-01-07  16.510000  16.549999  16.480000  16.480000      9
2015-01-08  16.351000  16.351000  16.351000  16.351000      0
...               ...        ...        ...        ...    ...
2024-12-24  29.974001  29.974001  29.974001  29.974001     66
2024-12-26  30.047001  30.115000  29.980000  30.014999    109
2024-12-27  29.655001  29.934999  29.605000  29.934999    502
2024-12-30  29.106001  29.719999  28.990000  29.660000    200
2024-12-31  28.940001  29.170000  28.940001  29.135000    172

[2513 rows x 5 columns]


In [15]:
def prepare_silver_news(csv_path, start, end):
    # 1. อ่านไฟล์โดยใช้ ';' เป็นตัวคั่นตามข้อมูลจริง
    df = pd.read_csv(csv_path, sep=';', on_bad_lines='skip')
    
    # 2. เลือกเฉพาะคอลัมน์ที่ต้องการใช้จริง (ชื่อตามในไฟล์ของคุณ)
    # คอลัมน์ในไฟล์คือ 'timestamp' และ 'headlines'
    df = df[['timestamp', 'headlines']]
    
    # 3. แปลงวันที่และกรองช่วงเวลา
    df['timestamp'] = pd.to_datetime(df['timestamp']).dt.date
    start_dt = pd.to_datetime(start).date()
    end_dt = pd.to_datetime(end).date()
    
    mask = (df['timestamp'] >= start_dt) & (df['timestamp'] <= end_dt)
    df_final = df.loc[mask].copy().sort_values('timestamp')
    
    return df_final

In [16]:
silver_csv_path = "data/final_silver_data.csv"
df_news = prepare_silver_news(silver_csv_path, start_date, end_date)

# แสดงผลแบบตารางสวยงาม
print("--- Silver News Data (Timestamp & Headlines) ---")
display(df_news)

--- Silver News Data (Timestamp & Headlines) ---


,timestamp,headlines
3592,2015-01-02,Blaze at Libya’s Largest Oil Port Extinguished...
3593,2015-01-05,Sony Head Thanks Supporters in Attack / Jury S...
3594,2015-01-06,Consumer Electronics Show Roundup: News Digest...
3595,2015-01-07,FAA Sets Standards for Airline Safety-Data Ana...
3596,2015-01-08,Mexico’s Televisa Sells Mobile Stake / China I...
...,...,...
6100,2024-12-24,The WSJ Dollar Index Edges Up to 102.37 / Tech...
6101,2024-12-26,The Epic Mess at TGI Fridays / The Celebrity A...
6102,2024-12-27,Yellen Says ‘Extraordinary Measures’ to Avoid ...
6103,2024-12-30,"Trump Endorses Mike Johnson for House Speaker,..."


## 2. Sentiment News

In [7]:
!pip install vaderSentiment

Defaulting to user installation because normal site-packages is not writeable
                                              0.0/126.0 kB ? eta -:--:--
                                              0.0/126.0 kB ? eta -:--:--
                                              0.0/126.0 kB ? eta -:--:--
     ---                                      10.2/126.0 kB ? eta -:--:--
     ---                                      10.2/126.0 kB ? eta -:--:--
     ---                                      10.2/126.0 kB ? eta -:--:--
     ---------                             30.7/126.0 kB 187.9 kB/s eta 0:00:01
     ------------                          41.0/126.0 kB 164.3 kB/s eta 0:00:01
     ------------                          41.0/126.0 kB 164.3 kB/s eta 0:00:01
     ------------------                    61.4/126.0 kB 172.4 kB/s eta 0:00:01
     ------------------                    61.4/126.0 kB 172.4 kB/s eta 0:00:01
     ------------------                    61.4/126.0 kB 172.4 kB/s eta 0:00:01
 

In [8]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from tqdm import tqdm
from IPython.display import display

In [ ]:
analyzer = SentimentIntensityAnalyzer()

new_words = {
    'bullish': 2.0,  'bearish': -2.0,
    'rally': 1.5,    'plummet': -1.5,
    'surge': 1.5,    'plunge': -1.5,
    'gain': 1.0,     'loss': -1.0,
    'high': 0.5,     'low': -0.5
}
analyzer.lexicon.update(new_words)

def analyze_silver_sentiment_vader(df):
    # เช็คว่ามีคอลัมน์ headlines ไหม
    if 'headlines' not in df.columns:
        print("ไม่พบคอลัมน์ 'headlines'")
        return df

    print("Running VADER Sentiment Analysis...")

    scores = []
    headlines = df['headlines'].fillna("").astype(str).tolist()
    
    for text in tqdm(headlines):
        # คำนวณคะแนน
        vs = analyzer.polarity_scores(text)
        scores.append(vs['compound'])

    df['sentiment_score'] = scores
    
    df['sentiment_value'] = df['sentiment_score'].apply(
        lambda x: 1 if x >= 0.05 else (-1 if x <= -0.05 else 0)
    )
    
    return df

In [20]:
df_analyzed = analyze_silver_sentiment_vader(df_news.copy())

print("\n--- VADER Sentiment Results ---")
display(df_analyzed[['timestamp', 'headlines', 'sentiment_score', 'sentiment_value']])

Running VADER Sentiment Analysis...


100%|██████████| 2513/2513 [00:02<00:00, 902.49it/s] 


--- VADER Sentiment Results ---


,timestamp,headlines,sentiment_score,sentiment_value
3592,2015-01-02,Blaze at Libya’s Largest Oil Port Extinguished...,-0.7351,-1
3593,2015-01-05,Sony Head Thanks Supporters in Attack / Jury S...,-0.3612,-1
3594,2015-01-06,Consumer Electronics Show Roundup: News Digest...,-0.8316,-1
3595,2015-01-07,FAA Sets Standards for Airline Safety-Data Ana...,-0.6808,-1
3596,2015-01-08,Mexico’s Televisa Sells Mobile Stake / China I...,-0.6124,-1
...,...,...,...,...
6100,2024-12-24,The WSJ Dollar Index Edges Up to 102.37 / Tech...,-0.8248,-1
6101,2024-12-26,The Epic Mess at TGI Fridays / The Celebrity A...,0.2500,1
6102,2024-12-27,Yellen Says ‘Extraordinary Measures’ to Avoid ...,0.3818,1
6103,2024-12-30,"Trump Endorses Mike Johnson for House Speaker,...",0.8316,1
